In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

### Scenario:
You're given `employee_fixed_width.txt` — a **fixed-width** text export, the kind of format still common from legacy mainframe/ERP systems that don't emit CSV or JSON at all. There are no delimiters; each field occupies an exact, known character position in every line. You have to impose the structure yourself using substring positions.

**Layout (0-indexed character positions, exclusive end):**

| Field | Start | End | Width |
| :--- | :--- | :--- | :--- |
| **emp_id** | 0 | 5 | 5 |
| **name** | 5 | 20 | 15 |
| **department** | 20 | 30 | 10 |
| **salary** | 30 | 38 | 8 |

Fields are space-padded on the right (except `salary`, which is zero-padded on the left, e.g. `00090000` = `90000`).

**Problem:**

- Read the file as plain text (`spark.read.text`), giving you one raw line per row.
- Use `substring()` to slice out `emp_id`, `name`, `department`, and `salary` at their fixed positions from the layout above.
- Trim trailing whitespace from `name` and `department`.
- Cast `salary` to an integer (the leading zeros should resolve naturally once cast).
- Write the resulting structured DataFrame out as **Parquet, partitioned by `department`**.
- Read the Parquet output back and display it, ordered by `emp_id` ascending, to confirm the round trip worked.

**Expected Output**

| emp_id | name | department | salary |
| :--- | :--- | :--- | :--- |
| E0001 | Alice Johnson | Sales | 90000 |
| E0002 | Bob Singh | Sales | 85000 |
| E0003 | Carol Mehta | Engg | 120000 |
| E0004 | David Kim | Engg | 115000 |
| E0005 | Eve Torres | HR | 70000 |

In [0]:
emp_data = spark.read.text("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/employee_fixed_width.txt")

emp_df = emp_data.withColumns(
    {
        "emp_id": substring(col("value"), 1, 5),
        "name": rtrim(substring(col("value"), 6, 15)),
        "department": rtrim(substring(col("value"), 21, 10)),
        "salary": (substring(col("value"), 31, 8)).cast(IntegerType())
    }
)

emp_df.drop("value").write.format("parquet").mode("overwrite") 
    .partitionBy("department") 
    .save("/Volumes/pyspark_practice/default/volumes/pyspark_practice_files/employees_partitioned")

emp_parquet_df = spark.read.format("parquet").load("/Volumes/pyspark_practice/default/volumes/pyspark_practice_files/employees_partitioned")
emp_parquet_df.orderBy(col("emp_id").asc()).show()

### Scenario:
You're given `customer_orders_nested.json` — a deeper, multi-level nested structure than you've handled before. Each order has: a nested `customer` **struct** (which itself contains a nested `address` struct), and an `items` **array of structs**, where each item struct itself contains a nested **array of strings** (`tags`). This kind of doubly-nested shape (array inside a struct inside an array) is very common in real event payloads — think product catalogs, order systems, or API responses.

**Problem — do both parts:**

**Part 1 — Order summary (flatten structs, aggregate array):**
- Read the JSON and check `.printSchema()` to see how the nested `customer.address` struct and the `items` array are represented.
- Produce one row per `order_id` with: `customer_id`, `customer_name`, `city`, `state` (all pulled out of the nested struct using dot notation), and `total_order_value` (sum of `qty` × `price` across that order's `items` — you'll need to explode `items` to compute this, then aggregate back to one row per order).

**Part 2 — Tag frequency (double explode):**
- Starting fresh from the same source, explode `items`, then explode the `tags` array **within** each item, to get one row per individual tag.
- Count how many times each tag appears across all orders.
- Order by count descending, then tag ascending.
- Note: `O2`'s single item has an **empty tags array** (`[]`) — it should contribute zero rows to this output, not a null/empty row.

**Expected Output — Part 1 (order summary)**

| order_id | customer_id | customer_name | city | state | total_order_value |
| :--- | :--- | :--- | :--- | :--- | :--- |
| O1 | C001 | Alice | Hyderabad | TS | 45.0 |
| O2 | C002 | Bob | Chennai | TN | 15.0 |

**Expected Output — Part 2 (tag frequency)**

| tag | tag_count |
| :--- | :--- |
| new | 1 |
| premium | 1 |
| sale | 1 |

In [0]:
## Part 1
cust_orders_json = spark.read.format("json").load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/customer_orders_nested.json")

"""
cust_orders_json.printSchema()
root
 |-- customer: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- state: string (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- product: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- tags: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |-- order_id: string (nullable = true)
"""

cust_orders_df = cust_orders_json.withColumn(
    "items_info", explode(col("items"))
)
cust_orders_df = cust_orders_df.select(
    "order_id",
    "customer.customer_id",
    "customer.name",
    "customer.address.city",
    "customer.address.state",
    "items_info.qty",
    "items_info.price"
)
cust_orders_df = cust_orders_df.withColumn("order_value", col("qty") * col("price")) \
    .groupBy("order_id", "customer_id", "name", "city", "state") \
    .agg(sum("order_value").alias("total_order_value"))

cust_orders_df.orderBy(col("order_id").asc()).show()


In [0]:
## Part 2
df = cust_orders_json.withColumn("items_info", explode(col("items")))
orders_with_tag_df = df.withColumn("tag", explode("items_info.tags"))

tags_count_df = orders_with_tag_df.groupBy(col("tag")).agg(count("order_id").alias("tag_count"))
tags_count_df.orderBy(col("tag_count").desc(), col("tag").asc()).show()